# 02 — EDA: Feature Exploration
**Inputs**
- `data/raw/Zeitreihen_WKA_2025_2026.xlsx` — power production (target)
- `data/raw/Zeitreihen_Wetterprognosen_2025_2026.xlsx` — weather forecasts
- `data/raw/water_data/Q_Rothenstein_15min.csv` — discharge 15-min (Saale upstream)
- `data/raw/water_data/Q_Rothenstein_1D.csv` — discharge daily (long history)

**Goals**
1. Timezone alignment audit — confirm UTC everywhere before merging
2. Discharge Q vs power — correlation at 15-min and daily
3. Weather features vs power — solar, temperature, wind
4. Lag feature validation — lag 1, 4, 96 signal strength
5. Candidate feature summary


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4)})


## 1 · Load All Sources

In [ ]:
# ── Power (target) ──────────────────────────────────────────────────────────
raw = pd.read_excel("../data/raw/Zeitreihen_WKA_2025_2026.xlsx", header=None)
wka = raw.iloc[5:].copy()
wka.columns = ["Datum_von", "Datum_bis", "Unterpreilipp_kW", "Burgau_kW"]
wka["Datum_von"] = (
    pd.to_datetime(wka["Datum_von"], dayfirst=True)
      .dt.tz_localize("Europe/Berlin", ambiguous="infer", nonexistent="shift_forward")
      .dt.tz_convert("UTC")
)
wka["Unterpreilipp_kW"] = pd.to_numeric(wka["Unterpreilipp_kW"], errors="coerce")
wka["Burgau_kW"]        = pd.to_numeric(wka["Burgau_kW"],        errors="coerce")
wka = wka.drop_duplicates(subset="Datum_von").set_index("Datum_von").drop(columns=["Datum_bis"])
wka = wka.sort_index()
print("Power  :", wka.index.min(), "→", wka.index.max(), f"({len(wka):,} rows)")


In [ ]:
# ── Weather forecasts ───────────────────────────────────────────────────────
raw_wx = pd.read_excel("../data/raw/Zeitreihen_Wetterprognosen_2025_2026.xlsx", header=None)
wx = raw_wx.iloc[6:].copy()
wx.columns = ["Datum_von", "Datum_bis", "solar_W_m2", "temp_C", "wind_ms", "wind_dir_deg"]
wx["Datum_von"] = (
    pd.to_datetime(wx["Datum_von"], dayfirst=True)
      .dt.tz_localize("Europe/Berlin", ambiguous="infer", nonexistent="shift_forward")
      .dt.tz_convert("UTC")
)
for col in ["solar_W_m2", "temp_C", "wind_ms", "wind_dir_deg"]:
    wx[col] = pd.to_numeric(wx[col], errors="coerce")
wx = wx.drop_duplicates(subset="Datum_von").set_index("Datum_von").drop(columns=["Datum_bis"])
wx = wx.sort_index()
print("Weather:", wx.index.min(), "→", wx.index.max(), f"({len(wx):,} rows)")


In [ ]:
# ── Discharge Q — 15-min ────────────────────────────────────────────────────
q15 = pd.read_csv("../data/raw/water_data/Q_Rothenstein_15min.csv")
q15["phenomenonTime"] = pd.to_datetime(q15["phenomenonTime"], utc=True)
q15 = q15.set_index("phenomenonTime").sort_index()
q15.columns = ["Q_m3s"]
print("Q 15min:", q15.index.min(), "→", q15.index.max(), f"({len(q15):,} rows)")

# ── Discharge Q — daily (long history) ──────────────────────────────────────
q1d = pd.read_csv("../data/raw/water_data/Q_Rothenstein_1D.csv")
q1d["phenomenonTime"] = pd.to_datetime(q1d["phenomenonTime"], utc=True)
q1d = q1d.set_index("phenomenonTime").sort_index()
q1d.columns = ["Q_m3s"]
print("Q daily:", q1d.index.min(), "→", q1d.index.max(), f"({len(q1d):,} rows)")


## 2 · Timezone Alignment Audit

All sources must be UTC before merging.  Print the tz attribute of every index.


In [ ]:
for name, df in [("Power (wka)", wka), ("Weather (wx)", wx),
                  ("Q 15min (q15)", q15), ("Q daily (q1d)", q1d)]:
    print(f"{name:20s}  tz = {df.index.tz}")
print()
print("✓ All UTC — safe to merge.")


## 3 · Discharge Q vs Power

In [ ]:
# Daily merge — long history
power_daily = wka[["Unterpreilipp_kW", "Burgau_kW"]].resample("D").mean()
q_daily     = q1d.resample("D").mean()
merged_1d   = power_daily.join(q_daily, how="inner")
merged_1d   = merged_1d[(merged_1d["Unterpreilipp_kW"] > 0) & (merged_1d["Burgau_kW"] > 0)]

print("Daily merge shape:", merged_1d.shape)
for col in ["Unterpreilipp_kW", "Burgau_kW"]:
    r = merged_1d[col].corr(merged_1d["Q_m3s"])
    print(f"  r(Q, {col}) = {r:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, ["Unterpreilipp_kW", "Burgau_kW"], ["steelblue", "darkorange"]):
    ax.scatter(merged_1d["Q_m3s"], merged_1d[col], alpha=0.3, s=8, color=color)
    ax.set_xlabel("Discharge Q (m³/s)")
    ax.set_ylabel("kW (daily mean)")
    r = merged_1d[col].corr(merged_1d["Q_m3s"])
    ax.set_title(f"{col}  r = {r:.3f}")
plt.suptitle("Daily Q vs Power — Rothenstein gauge")
plt.tight_layout()
plt.show()


In [ ]:
# 15-min merge — shorter but model-resolution
merged_15 = wka.join(q15, how="inner")
merged_15 = merged_15[(merged_15["Unterpreilipp_kW"] > 0)]
print("15-min merge shape:", merged_15.shape)
for col in ["Unterpreilipp_kW", "Burgau_kW"]:
    r = merged_15[col].corr(merged_15["Q_m3s"])
    print(f"  r(Q, {col}) at 15-min = {r:.3f}")


## 4 · Weather Features vs Power

In [ ]:
merged_wx = wka.join(wx, how="inner")
merged_wx = merged_wx[merged_wx["Unterpreilipp_kW"] > 0]

features = ["solar_W_m2", "temp_C", "wind_ms"]
print("Pearson correlations with Unterpreilipp_kW:")
for f in features:
    r = merged_wx["Unterpreilipp_kW"].corr(merged_wx[f])
    print(f"  {f:20s}  r = {r:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    ax.scatter(merged_wx[feat], merged_wx["Unterpreilipp_kW"],
               alpha=0.15, s=4, color="steelblue")
    r = merged_wx["Unterpreilipp_kW"].corr(merged_wx[feat])
    ax.set_xlabel(feat)
    ax.set_ylabel("Unterpreilipp kW")
    ax.set_title(f"r = {r:.3f}")
plt.suptitle("Weather features vs Unterpreilipp power (non-zero)")
plt.tight_layout()
plt.show()


## 5 · Lag Feature Validation

In [ ]:
# Test lags 1, 4, 96 for Unterpreilipp
series = wka["Unterpreilipp_kW"].dropna()
series = series[series > 0]

lag_results = []
for lag in [1, 4, 8, 96, 192, 672]:  # 15min, 1h, 2h, 1day, 2days, 1week
    r = series.corr(series.shift(lag))
    lag_results.append({"lag": lag, "offset": f"{lag*15} min", "r": round(r, 3)})

lag_df = pd.DataFrame(lag_results)
print(lag_df.to_string(index=False))


In [ ]:
# Bar chart of lag correlations
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(lag_df["offset"], lag_df["r"], color="steelblue", edgecolor="white")
ax.bar_label(bars, fmt="%.3f", fontsize=9)
ax.set_xlabel("Lag")
ax.set_ylabel("Pearson r with Unterpreilipp_kW")
ax.set_title("Lag feature correlation — Unterpreilipp (non-zero periods)")
ax.axhline(0, color="black", lw=0.8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 6 · Candidate Feature Summary

| Feature | Source | Availability at forecast time | Expected signal |
|---|---|---|---|
| `lag_1` (t-15min) | Power history | ✅ known | High (autocorrelation) |
| `lag_4` (t-1h) | Power history | ✅ known | High |
| `lag_96` (t-1day) | Power history | ✅ known | **Highest** — seasonal persistence |
| `Q_m3s` upstream | Gauge API | ✅ known (real-time) | High (physical cause) |
| `solar_W_m2` | DWD forecast | ✅ forecast | Low–medium |
| `temp_C` | DWD forecast | ✅ forecast | Low |
| `wind_ms` | DWD forecast | ✅ forecast | Low |
| `hour`, `month`, `weekday` | Calendar | ✅ always known | Medium (seasonality) |
| `plant_offline` flag | Derived from zeros | ✅ if pre-announced | Burgau only |

**Recommendation:** Q and lag_96 are the two most important features.  
Weather features add marginal signal but complete the explainability story for the jury.
